## Vectorless RAG with PageIndex

Standard RAG splits a document into chunks, embeds them, and retrieves by cosine similarity. **Vectorless RAG** keeps the document whole and instead builds a hierarchical *table-of-contents tree* over it. At query time, an LLM **navigates the tree** to find the right section — the way a human uses an index — and pulls the underlying text directly. No embeddings, no chunk fragmentation, and the retrieval path is interpretable end-to-end.

This notebook uses [PageIndex](https://pageindex.ai) (`pip install pageindex`), a hosted service that builds the tree and exposes a retrieval API. We'll index a paper, inspect the tree, and let a `create_agent` answer questions over it via two tools: `get_outline` and `search_doc`. Background reading: [PageIndex intro blog](https://pageindex.ai/blog/pageindex-intro).

### Setup

PageIndex is a hosted API, so you need a `PAGEINDEX_API_KEY` from [pageindex.ai](https://pageindex.ai) in your `.env`. We'll also use OpenAI for the LangChain agent in the later cells.

In [ ]:
# !pip install pageindex requests   # uncomment if running outside uv
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["PAGEINDEX_API_KEY"] = os.getenv("PAGEINDEX_API_KEY")

assert os.environ["PAGEINDEX_API_KEY"], "Set PAGEINDEX_API_KEY in your .env"
assert os.environ["OPENAI_API_KEY"], "Set OPENAI_API_KEY in your .env"

### 1. Grab a sample PDF

We'll use the *Attention Is All You Need* paper as our knowledge source — short, well-structured, and famously the basis for the Transformer architecture. The download is cached so re-runs are fast.

In [ ]:
import os, requests

PDF_URL = "https://arxiv.org/pdf/1706.03762.pdf"
PDF_PATH = "sample.pdf"

if not os.path.exists(PDF_PATH):
    r = requests.get(PDF_URL, timeout=60)
    r.raise_for_status()
    with open(PDF_PATH, "wb") as f:
        f.write(r.content)
    print(f"Downloaded {len(r.content) / 1024:.0f} KB to {PDF_PATH}")
else:
    print(f"{PDF_PATH} already exists ({os.path.getsize(PDF_PATH) / 1024:.0f} KB)")

### 2. Submit the document for indexing

`PageIndexClient.submit_document(file_path)` uploads the PDF and starts tree generation. It returns immediately with a `doc_id` — indexing happens asynchronously on the server, so we'll poll until it's ready.

In [ ]:
from pageindex import PageIndexClient

client = PageIndexClient(api_key=os.environ["PAGEINDEX_API_KEY"])
submission = client.submit_document(file_path=PDF_PATH)
doc_id = submission["doc_id"]
print("doc_id:", doc_id)

### 3. Wait for indexing to complete

`is_retrieval_ready(doc_id)` returns `True` once the tree has been built and the document is queryable. For a 15-page paper this typically takes 10–30 seconds.

In [ ]:
import time

deadline = time.time() + 180
while time.time() < deadline:
    if client.is_retrieval_ready(doc_id):
        print("Ready!")
        break
    print("...still indexing")
    time.sleep(5)
else:
    raise TimeoutError("Indexing took longer than 3 minutes")

### 4. Inspect the tree

This is the heart of the vectorless approach. `get_tree(doc_id, node_summary=True)` returns a JSON tree of section titles, page numbers, and (because we asked) short summaries. An LLM can read this tree and decide *which node* contains the answer to a question — the same way you'd skim a textbook's contents page.

In [ ]:
import json

tree = client.get_tree(doc_id, node_summary=True)
print(json.dumps(tree, indent=2)[:2500])

### 5. Try a direct retrieval

Before we wire it into an agent, let's see what raw retrieval looks like. `submit_query` returns a `retrieval_id` (again async); polling `get_retrieval` returns the matched nodes once ready.

In [ ]:
import time, json

def run_query(query: str, timeout: int = 60) -> dict:
    submission = client.submit_query(doc_id=doc_id, query=query)
    retrieval_id = submission["retrieval_id"]
    deadline = time.time() + timeout
    while time.time() < deadline:
        result = client.get_retrieval(retrieval_id)
        if result.get("status") in ("completed", "failed"):
            return result
        time.sleep(2)
    raise TimeoutError(f"Retrieval {retrieval_id} did not finish")

result = run_query("What is multi-head attention?")
print(json.dumps(result, indent=2)[:2000])

### 6. Wrap PageIndex as LangChain tools

Now we plug it into the agent pattern from notebooks 1 and 3. Two tools is enough: one to fetch the outline, one to search. The agent decides when to call which — that's the LLM doing the *navigation* step that vectorless RAG is built around.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

@tool
def get_outline() -> str:
    """Return the hierarchical outline (table of contents) of the indexed PDF as JSON."""
    return json.dumps(client.get_tree(doc_id, node_summary=True))[:6000]

@tool
def search_doc(query: str) -> str:
    """Search the indexed PDF for content relevant to a natural-language query. Returns matching nodes with their text."""
    return json.dumps(run_query(query))[:6000]

agent = create_agent(
    model="gpt-4o-mini",
    tools=[get_outline, search_doc],
    system_prompt=(
        "You answer questions about an indexed research paper. "
        "First call get_outline to see the section structure, "
        "then call search_doc with a focused query, "
        "then answer concisely with citations to section names."
    ),
)

### 7. Ask a real question

The agent should call `get_outline` first, pick a relevant section, then call `search_doc`, and finally synthesise an answer. The whole retrieval path is visible in `result['messages']` — that's the interpretability win over embedding-based RAG.

In [ ]:
result = agent.invoke({
    "messages": [HumanMessage(content="What is multi-head attention and why was it introduced?")]
})
print(result["messages"][-1].content)

In [ ]:
for msg in result["messages"]:
    cls = msg.__class__.__name__
    if getattr(msg, "tool_calls", None):
        for tc in msg.tool_calls:
            print(f"{cls} -> tool {tc['name']}({tc['args']})")
    else:
        print(f"{cls}: {str(msg.content)[:160]}")

### Recap: when to choose vectorless vs vector RAG

- **Vectorless wins** for long, structured documents (financial filings, contracts, papers, manuals) where hierarchy is meaningful and you want to *see* why a section was chosen.
- **Vector RAG wins** for many small documents, fuzzy semantic matching across a corpus, or when you need sub-second latency.
- PageIndex also ships an [MCP server](https://github.com/VectifyAI/pageindex-mcp) — once you've finished the bootcamp, you can plug the same indexed document into Claude Code or any MCP-compatible client without re-indexing.